# Lc0 evaluation of every legal move (version 2)

For each sampled position and each legal move, this notebook asks Leela Chess Zero how good the position is **after** that move, at a ladder of increasing node counts.

- Positions come from `optimal_positions.txt`, taken in a fixed, seeded order that keeps the piece-count mix of the whole file. Wherever the run stops, the finished positions are a fair sample.
- Each move gets its own search, read out at 25 node counts spaced evenly on a log scale from 1 to 10,000 (1, 2, 3, 4, 5, 7, 10, 15, 22 ... 3162, 4642, 6813, 10000). Lc0 keeps its search tree between the readings, so the readings cost about as much as the top level alone, plus roughly 0.02 s per reading.
- The top level is `TOP_NODES` (10,000 by default). With a fixed top level the run takes positions in their seeded order until the time budget (8.5 hours) is nearly used up; on two T4s that is a few hundred positions. Setting `TOP_NODES = None` lets a short calibration choose the highest top level that fits `TARGET_POSITIONS`.
- Results are written to disk after every position, and the run stops by itself before the time budget ends.

**To run on Kaggle:** accelerator *GPU T4 x2*, internet on, the positions dataset attached, then *Save Version → Save & Run All*.

**Output** (folder `/kaggle/working/lc0_run`, also zipped as `lc0_run.zip`):
`lc0_evals.csv`, `positions_analyzed.txt`, `positions_order.csv`, `run_info.json`. The last cell explains the columns.

In [ ]:
# ---- Settings: change numbers here, nowhere else --------------------------------
import time
T0 = time.time()               # the time budget counts from this cell, so building Lc0 is included

TOTAL_BUDGET_HOURS = 8.5       # no new position is started once this is nearly used up (Kaggle stops a session at about 9 hours)
TOP_NODES          = 10_000    # nodes per move at the top of the ladder; None = let a calibration choose
LEVELS_PER_DECADE  = 6         # log-spaced ladder from 1 node up to the top: 6 per factor of 10 = 25 levels up to 10,000 (8 would give 33)
TARGET_POSITIONS   = 1000      # used only when TOP_NODES = None: the calibration picks the highest top level that fits this many
MAX_TOP_NODES      = 10_000    # used only when TOP_NODES = None: highest top level the calibration may pick
SEED               = 20261110  # fixes the order of the positions; do not change between runs
RESUME_DIR         = ""        # folder holding an earlier run's four output files, to continue that run

# With TOP_NODES fixed, the run takes positions in their seeded order until the time budget ends.
# Quick trial before a full run: set TOTAL_BUDGET_HOURS = 1.0; the log shows the seconds each position takes.

In [ ]:
!apt-get update
!apt-get -y install git python3-pip gcc-10 g++-10 zlib1g zlib1g-dev
!pip install --upgrade meson ninja

In [ ]:
!git clone -b release/0.32 https://github.com/LeelaChessZero/lc0.git

In [ ]:
%cd /kaggle/working/lc0
!CC=gcc-10 CXX=g++-10 meson setup build/release --buildtype=release -Dbackend_max_links=1
!ninja -C build/release lc0 -j 1

In [ ]:
%cd /kaggle/working/lc0/build/release
!rm -f weights.pb.gz
!wget -q --show-progress https://storage.lczero.org/files/networks-contrib/BT4-1024x15x32h-swa-6147500-policytune-332.pb.gz -O weights.pb.gz

In [ ]:
!pip install chess

In [ ]:
# =============================================================================
# Lc0 per-move evaluation.  Settings come from the first cell of this notebook.
#
# For every sampled position and every legal move, the position AFTER the move
# is searched up a ladder of node counts.  Lc0 keeps its search tree between
# the steps of one ladder, so the whole ladder costs only its top level.
# Output (folder lc0_run/):
#   lc0_evals.csv           one row per position, move and ladder level
#   positions_analyzed.txt  the positions that were finished (for Stockfish)
#   positions_order.csv     the fixed, seeded order positions are taken in
#   run_info.json           settings, ladder, calibration table, timings
# =============================================================================
import csv
import glob
import hashlib
import json
import os
import random
import shutil
import subprocess
import threading
import time
import zipfile

import chess
import chess.engine

# ---- Paths (Kaggle defaults; environment variables override them elsewhere) --
LC0_PATH = os.environ.get("LC0_PATH", "/kaggle/working/lc0/build/release/lc0")
WEIGHTS_PATH = os.environ.get("LC0_WEIGHTS", "/kaggle/working/lc0/build/release/weights.pb.gz")
OUT_DIR = os.environ.get("LC0_OUT_DIR", "/kaggle/working/lc0_run")
POSITIONS_TXT = os.environ.get("LC0_POSITIONS", "")  # empty: look in /kaggle/input, then GitHub
BACKEND = os.environ.get("LC0_BACKEND", "cuda-fp16")
N_ENGINES = int(os.environ.get("LC0_ENGINES", "0"))  # 0: one engine per GPU
ENGINE_THREADS = int(os.environ.get("LC0_THREADS", "2"))
REPO_URL = "https://github.com/likutamukonda/chessgametreepaper"

MINIBATCH = 256              # largest batch sent to the GPU (same as the first run of this project)
CAP_BATCH_AT_STEP = True     # keep each batch within the nodes a ladder step asks for, so the
                             # node counts Lc0 reaches stay close to the ladder's targets
SAFETY = 1.15                # head-room when projecting the run time
CALIBRATION_FLOOR = 100      # the calibration never picks a top level below this many nodes per move
FINISH_MARGIN_S = int(os.environ.get("LC0_FINISH_MARGIN", "300"))   # seconds kept free at the end for the last position and saving
CALIBRATION_POSITIONS = int(os.environ.get("LC0_CALIBRATION", "6"))
PIECE_BANDS = [(2, 6), (7, 11), (12, 16), (17, 21), (22, 26), (27, 32)]

CSV_PATH = os.path.join(OUT_DIR, "lc0_evals.csv")
DONE_PATH = os.path.join(OUT_DIR, "positions_analyzed.txt")
ORDER_PATH = os.path.join(OUT_DIR, "positions_order.csv")
INFO_PATH = os.path.join(OUT_DIR, "run_info.json")
CSV_HEADER = ["pos_id", "move", "level", "target_nodes", "nodes", "wl",
              "w", "d", "l", "mate", "terminal", "secs"]
DONE_HEADER = "pos_id\tfile_index\tlegal_moves\tfen\n"


def say(*parts):
    print(*parts, flush=True)


def clock(seconds):
    seconds = int(max(0, seconds))
    return f"{seconds // 3600}:{seconds % 3600 // 60:02d}:{seconds % 60:02d}"


def node_ladder(top, per_decade):
    """Node counts 1 ... top, evenly spaced on a log scale (per_decade levels per factor of 10).

    1 node is the network alone, with no search.  Every level is a different whole
    number: where rounding would repeat one at the low end (1, 1.3, 1.8, 2.4 ...),
    the next whole number is used.  With per_decade = 8 and top = 1000 this gives 25 levels:
    1 2 3 4 5 6 7 8 10 13 18 24 32 42 56 75 100 133 178 237 316 422 562 750 1000
    """
    levels, k = [1], 1
    while True:
        n = max(levels[-1] + 1, int(round(10 ** (k / per_decade))))
        if n > top:
            break
        levels.append(n)
        k += 1
    if levels[-1] < top:
        levels.append(int(top))
    return levels


# ---- Positions and their fixed order -----------------------------------------
def find_positions_file():
    if POSITIONS_TXT:
        if not os.path.exists(POSITIONS_TXT):
            raise FileNotFoundError(POSITIONS_TXT)
        return POSITIONS_TXT
    hits = sorted(glob.glob("/kaggle/input/**/optimal_positions.txt", recursive=True))
    if hits:
        return hits[0]
    say("Positions file not found under /kaggle/input; fetching it from GitHub ...")
    base = os.path.join(os.path.dirname(OUT_DIR.rstrip("/")) or ".", "_positions_repo")
    if not os.path.isdir(base):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, base], check=True)
    for archive in sorted(glob.glob(os.path.join(base, "**", "*.zip"), recursive=True)):
        with zipfile.ZipFile(archive) as zf:
            for name in zf.namelist():
                if name.endswith("optimal_positions.txt"):
                    target = os.path.join(base, "_unzipped")
                    zf.extract(name, target)
                    return os.path.join(target, name)
    raise FileNotFoundError("optimal_positions.txt was not found in /kaggle/input or in the GitHub repository")


def piece_band(fen):
    pieces = sum(ch.isalpha() for ch in fen.split(" ", 1)[0])
    for band, (low, high) in enumerate(PIECE_BANDS):
        if low <= pieces <= high:
            return band, pieces
    return len(PIECE_BANDS) - 1, pieces


def build_order(fens, seed):
    """Seeded order in which every prefix holds the piece-count bands in proportion.

    Positions are shuffled inside each band and then interleaved, so wherever
    the run stops, the finished positions are a fair sample of the whole file.
    """
    rng = random.Random(seed)
    bands = [[] for _ in PIECE_BANDS]
    for index, fen in enumerate(fens):
        bands[piece_band(fen)[0]].append(index)
    keyed = []
    for members in bands:
        rng.shuffle(members)
        offset = rng.random()
        keyed.extend(((rank + offset) / len(members), index) for rank, index in enumerate(members))
    keyed.sort()
    return [index for _, index in keyed]


def sha256_of(path):
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for block in iter(lambda: fh.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()


# ---- Engine -------------------------------------------------------------------
class Lc0Engine:
    def __init__(self, index):
        self.index = index
        self.engine = None
        self.name = ""
        self.start()

    def start(self):
        self.engine = chess.engine.SimpleEngine.popen_uci(LC0_PATH, timeout=900)
        options = {
            "WeightsFile": WEIGHTS_PATH,
            "Backend": BACKEND,
            "Threads": ENGINE_THREADS,
            "MinibatchSize": MINIBATCH,
            "ScoreType": "W-L",          # score = wins minus losses, reported to 4 decimals
            "SmartPruningFactor": 0.0,   # never stop a search before its node target
            "UCI_ShowWDL": True,
        }
        if BACKEND.startswith(("cuda", "cudnn")):
            options["BackendOptions"] = f"gpu={self.index}"
        missing = [name for name in options if name not in self.engine.options]
        if missing:
            raise RuntimeError(f"This Lc0 build has no option(s) {missing}")
        self.engine.configure(options)
        self.name = self.engine.id.get("name", "")
        # Loads the network (onto the GPU) before anything is timed.
        self.engine.analyse(chess.Board(), chess.engine.Limit(nodes=1),
                            game=("warm-up", self.index, time.time()))

    def restart(self):
        try:
            self.engine.close()
        except Exception:
            pass
        self.start()

    def close(self):
        try:
            self.engine.quit()
        except Exception:
            pass


def gpu_count():
    try:
        out = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True, timeout=60).stdout
        return max(1, sum(1 for line in out.splitlines() if line.strip().startswith("GPU ")))
    except Exception:
        return 1


def terminal_kind(board):
    if board.is_checkmate():
        return "checkmate"
    if board.is_stalemate():
        return "stalemate"
    if board.is_insufficient_material():
        return "insufficient_material"
    return ""


def read_info(info, mover):
    """Score of a finished search, from the point of view of the player who made the move."""
    score = info.get("score")
    if score is None:
        return None
    pov = score.pov(mover)
    value = pov.score(mate_score=1_000_000)
    if pov.is_mate():                      # proven win or loss
        wl, mate = (1.0 if value > 0 else -1.0), pov.mate()
    else:
        wl, mate = value / 10000.0, ""
    w = d = l = ""
    wdl = info.get("wdl")
    if wdl is not None:
        p = wdl.pov(mover)
        w, d, l = p.wins, p.draws, p.losses
    return wl, w, d, l, mate, info.get("nodes", "")


def analyse_position(eng, pos, ladder):
    """Search the position after each legal move up the whole node ladder."""
    board = chess.Board(pos["fen"])
    mover = board.turn
    game = ("position", pos["pos_id"], eng.index, time.time())  # new game: Lc0 starts clean
    rows, searches, started = [], 0, time.perf_counter()
    for move in board.legal_moves:
        child = board.copy(stack=False)
        child.push(move)
        terminal = terminal_kind(child)
        if terminal:                       # the move ends the game: no search needed
            wl = 1.0 if terminal == "checkmate" else 0.0
            w, d, l = (1000, 0, 0) if terminal == "checkmate" else (0, 1000, 0)
            for level, target in enumerate(ladder):
                rows.append([pos["pos_id"], move.uci(), level, target, 0, f"{wl:.4f}",
                             w, d, l, "", terminal, 0.0])
            continue
        t0, last, have = time.perf_counter(), None, 0
        for level, target in enumerate(ladder):
            # Same position and same game as the previous step, so Lc0 continues
            # its existing tree up to `target` nodes in total.
            batch = MINIBATCH
            if CAP_BATCH_AT_STEP:
                batch = max(1, min(MINIBATCH, (target - have) // ENGINE_THREADS))
            info = eng.engine.analyse(child, chess.engine.Limit(nodes=target), game=game,
                                      options={"MinibatchSize": batch})
            record = read_info(info, mover) or last
            if record is None:
                raise RuntimeError(f"Lc0 returned no score for {child.fen()}")
            last = record
            wl, w, d, l, mate, nodes = record
            if isinstance(nodes, int):
                have = max(have, nodes)
            rows.append([pos["pos_id"], move.uci(), level, target, nodes, f"{wl:.4f}",
                         w, d, l, mate, "", round(time.perf_counter() - t0, 4)])
        searches += 1
    return rows, searches, time.perf_counter() - started


# ---- Files ----------------------------------------------------------------------
def write_info(update):
    info = {}
    if os.path.exists(INFO_PATH):
        with open(INFO_PATH) as fh:
            info = json.load(fh)
    info.update(update)
    tmp = INFO_PATH + ".tmp"
    with open(tmp, "w") as fh:
        json.dump(info, fh, indent=1)
    os.replace(tmp, INFO_PATH)
    return info


def load_previous_run():
    """Continue an earlier run: returns (top_nodes, finished position ids) or None."""
    if RESUME_DIR and os.path.isdir(RESUME_DIR) and not os.path.exists(DONE_PATH):
        for name in os.listdir(RESUME_DIR):
            source = os.path.join(RESUME_DIR, name)
            if os.path.isfile(source):
                shutil.copy2(source, os.path.join(OUT_DIR, name))
    if not all(os.path.exists(p) for p in (DONE_PATH, INFO_PATH, CSV_PATH)):
        return None
    with open(INFO_PATH) as fh:
        info = json.load(fh)
    if not info.get("top_nodes"):
        return None
    for key, now in (("seed", SEED), ("levels_per_decade", LEVELS_PER_DECADE)):
        if info.get(key) != now:
            raise RuntimeError(f"The run in {OUT_DIR} used {key}={info.get(key)}, but this notebook has {now}")
    if info.get("ladder") != node_ladder(int(info["top_nodes"]), LEVELS_PER_DECADE):
        raise RuntimeError(f"The run in {OUT_DIR} used the node ladder {info.get('ladder')}, which is not the ladder "
                           "this notebook builds (a run made with an older version of the notebook cannot be continued)")
    with open(DONE_PATH) as fh:
        done = {int(line.split("\t")[0]) for line in fh if line[:1].isdigit()}
    # Drop rows of a position that was cut off before it was marked as finished.
    kept, dropped, tmp = 0, 0, CSV_PATH + ".tmp"
    with open(CSV_PATH, newline="") as src, open(tmp, "w", newline="") as dst:
        reader, writer = csv.reader(src), csv.writer(dst)
        writer.writerow(next(reader))
        for row in reader:
            if int(row[0]) in done:
                writer.writerow(row)
                kept += 1
            else:
                dropped += 1
    os.replace(tmp, CSV_PATH)
    say(f"Continuing an earlier run: {len(done)} positions already finished "
        f"({kept:,} rows kept, {dropped:,} unfinished rows dropped), top level {info['top_nodes']:,} nodes.")
    return int(info["top_nodes"]), done


# ---- The run --------------------------------------------------------------------
def run():
    os.makedirs(OUT_DIR, exist_ok=True)
    deadline = T0 + TOTAL_BUDGET_HOURS * 3600 - FINISH_MARGIN_S
    for path, what in ((LC0_PATH, "Lc0 program"), (WEIGHTS_PATH, "network file")):
        if not os.path.exists(path):
            raise FileNotFoundError(f"{what} not found at {path}")
    say(f"Network file: {os.path.getsize(WEIGHTS_PATH) / 1e6:.1f} MB")
    if BACKEND.startswith(("cuda", "cudnn")) and shutil.which("nvidia-smi") is None:
        raise RuntimeError("No GPU found. In the notebook settings choose the accelerator 'GPU T4 x2', then run again.")

    previous = load_previous_run()
    top, done = previous if previous else (TOP_NODES, set())

    # Positions, in their fixed order
    positions_file = find_positions_file()
    with open(positions_file) as fh:
        fens = [line.strip() for line in fh if line.strip()]
    order = build_order(fens, SEED)
    keep = min(len(order), max(5 * TARGET_POSITIONS, 5000))
    positions = []
    for pos_id, file_index in enumerate(order[:keep]):
        fen = fens[file_index]
        board = chess.Board(fen)
        # Positions that are not valid standard chess are left out.  In this file
        # those are Chess960 positions (castling rights that do not match where
        # the king and rooks stand).  Finished games are left out too.
        skip = ""
        if not board.is_valid():
            skip = "not standard chess"
        elif board.legal_moves.count() == 0:
            skip = "no legal moves"
        positions.append({"pos_id": pos_id, "file_index": file_index, "fen": fen,
                          "pieces": piece_band(fen)[1], "skip": skip,
                          "n_legal": 0 if skip else board.legal_moves.count()})
    with open(ORDER_PATH, "w", newline="") as fh:
        writer = csv.writer(fh)
        writer.writerow(["pos_id", "file_index", "pieces", "legal_moves", "skip", "fen"])
        for p in positions:
            writer.writerow([p["pos_id"], p["file_index"], p["pieces"], p["n_legal"], p["skip"], p["fen"]])
    skipped = sum(1 for p in positions[:TARGET_POSITIONS] if p["skip"])
    say(f"{len(fens):,} positions in {positions_file}; order fixed by seed {SEED}; "
        f"{skipped} of the first {min(TARGET_POSITIONS, len(positions))} are left out (not standard chess, or game over).")

    # Engines: one per GPU
    n_engines = N_ENGINES or gpu_count()
    engines = [Lc0Engine(i) for i in range(n_engines)]
    say(f"{n_engines} engine(s): {engines[0].name}, backend {BACKEND}, {ENGINE_THREADS} threads each.")

    write_info({"status": "running", "seed": SEED, "levels_per_decade": LEVELS_PER_DECADE,
                "target_positions": TARGET_POSITIONS, "total_budget_hours": TOTAL_BUDGET_HOURS,
                "engine": engines[0].name, "backend": BACKEND, "engines": n_engines,
                "engine_threads": ENGINE_THREADS, "minibatch": MINIBATCH,
                "score_type": "W-L", "smart_pruning_factor": 0.0,
                "weights_file": os.path.basename(WEIGHTS_PATH),
                "weights_bytes": os.path.getsize(WEIGHTS_PATH),
                "positions_file": positions_file, "positions_in_file": len(fens),
                "positions_sha256": sha256_of(positions_file),
                "python_chess": chess.__version__,
                "started_utc": time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime(T0))})

    lock = threading.Lock()
    new_csv = not os.path.exists(CSV_PATH)
    csv_file = open(CSV_PATH, "a", newline="")
    csv_writer = csv.writer(csv_file)
    done_file = open(DONE_PATH, "a")
    if new_csv:
        csv_writer.writerow(CSV_HEADER)
        done_file.write(DONE_HEADER)
    state = {"next": 0, "stop": False, "sec_per_search": None, "new": 0, "failed": [],
             "searches": 0, "search_secs": 0.0}

    def commit(pos, rows, searches, secs):
        with lock:
            csv_writer.writerows(rows)
            csv_file.flush()
            os.fsync(csv_file.fileno())
            done_file.write(f"{pos['pos_id']}\t{pos['file_index']}\t{pos['n_legal']}\t{pos['fen']}\n")
            done_file.flush()
            os.fsync(done_file.fileno())
            done.add(pos["pos_id"])
            state["new"] += 1
            if searches:
                per = secs / searches
                old = state["sec_per_search"]
                state["sec_per_search"] = per if old is None else 0.8 * old + 0.2 * per
                state["searches"] += searches
                state["search_secs"] += secs
            say(f"[{len(done):5d} done] position {pos['pos_id']:5d}  {pos['n_legal']:2d} moves  {secs:6.1f} s"
                f"  | used {clock(time.time() - T0)} of {clock(TOTAL_BUDGET_HOURS * 3600)}")

    # ---- Calibration: pick the top node level that fits the time budget ----------
    calibration = None
    if top is None:
        full = node_ladder(MAX_TOP_NODES, LEVELS_PER_DECADE)
        sample = [p for p in positions if p["n_legal"] > 0][:max(CALIBRATION_POSITIONS, n_engines)]
        say(f"Calibrating on {len(sample)} positions (ladder up to {MAX_TOP_NODES:,} nodes per move) ...")
        results, errors = {}, []

        def calibrate(eng, items):
            for p in items:
                try:
                    results[p["pos_id"]] = analyse_position(eng, p, full)
                except Exception as exc:
                    errors.append(exc)

        threads = [threading.Thread(target=calibrate, args=(eng, sample[i::n_engines]))
                   for i, eng in enumerate(engines)]
        for t in threads:
            t.start()
        for t in threads:
            t.join()
        if errors:
            raise errors[0]
        sums, counts = [0.0] * len(full), [0] * len(full)
        for rows, _, _ in results.values():
            for row in rows:
                if row[10] == "":
                    sums[row[2]] += row[11]
                    counts[row[2]] += 1
        secs_at = [s / max(1, c) for s, c in zip(sums, counts)]
        target_set = [p for p in positions if p["n_legal"] > 0][:TARGET_POSITIONS]
        moves_left = sum(p["n_legal"] for p in target_set if p["pos_id"] not in results)
        remaining = deadline - time.time()
        say(f"  {'nodes per move':>15} {'secs per move':>14} {'hours for ' + str(len(target_set)) + ' positions':>26}")
        table, choice = [], None
        for level, nodes in enumerate(full):
            hours = secs_at[level] * moves_left / n_engines * SAFETY / 3600
            table.append({"nodes": nodes, "secs_per_move": round(secs_at[level], 4), "hours": round(hours, 2)})
            fits = hours * 3600 <= remaining
            if nodes >= CALIBRATION_FLOOR:
                say(f"  {nodes:>15,} {secs_at[level]:>14.3f} {hours:>26.2f}{'   fits' if fits else ''}")
                if fits:
                    choice = nodes
        if choice is None:
            choice = next(n for n in full if n >= CALIBRATION_FLOOR)
            say(f"  Even {choice:,} nodes does not fit {len(target_set)} positions; using it and finishing fewer positions.")
        top = choice
        calibration = table
        say(f"Top level chosen: {top:,} nodes per move ({clock(remaining)} of the budget was left).")
        ladder = node_ladder(top, LEVELS_PER_DECADE)
        say(f"Ladder ({len(ladder)} levels): {ladder}")
        for p in sample:
            rows, searches, secs = results[p["pos_id"]]
            kept = [row for row in rows if row[2] < len(ladder)]
            share = secs_at[len(ladder) - 1] / secs_at[-1] if secs_at[-1] else 1.0
            commit(p, kept, searches, secs * share)
    ladder = node_ladder(top, LEVELS_PER_DECADE)
    write_info({"top_nodes": top, "ladder": ladder, "cap_batch_at_step": CAP_BATCH_AT_STEP,
                **({"calibration": calibration} if calibration else {})})
    if calibration is None:
        say(f"Ladder ({len(ladder)} levels, top {top:,} nodes per move): {ladder}")

    # ---- Main loop -----------------------------------------------------------------
    def take():
        with lock:
            while not state["stop"] and state["next"] < len(positions):
                p = positions[state["next"]]
                state["next"] += 1
                if p["pos_id"] in done or p["n_legal"] == 0:
                    continue
                estimate = (state["sec_per_search"] or 0.0) * p["n_legal"] * 1.15
                if time.time() + estimate > deadline:
                    state["stop"] = True
                    say(f"Time budget reached: not starting position {p['pos_id']}.")
                    return None
                return p
            state["stop"] = True
            return None

    def work(eng):
        while True:
            p = take()
            if p is None:
                return
            for attempt in (1, 2):
                try:
                    rows, searches, secs = analyse_position(eng, p, ladder)
                    commit(p, rows, searches, secs)
                    break
                except Exception as exc:
                    say(f"Position {p['pos_id']} failed on engine {eng.index} (attempt {attempt}): {exc!r}")
                    try:
                        eng.restart()
                    except Exception as exc2:
                        say(f"Engine {eng.index} could not be restarted: {exc2!r}")
                        with lock:
                            state["failed"].append(p["pos_id"])
                        return
            else:
                with lock:
                    state["failed"].append(p["pos_id"])

    threads = [threading.Thread(target=work, args=(eng,)) for eng in engines]
    for t in threads:
        t.start()
    for t in threads:
        t.join()

    # ---- Save ------------------------------------------------------------------------
    csv_file.close()
    done_file.close()
    for eng in engines:
        eng.close()
    info = write_info({"status": "finished", "positions_finished": len(done),
                       "positions_finished_this_session": state["new"],
                       "positions_failed": state["failed"],
                       "mean_secs_per_move_search": round(state["search_secs"] / max(1, state["searches"]), 4),
                       "elapsed_seconds": round(time.time() - T0),
                       "finished_utc": time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime())})
    archive = shutil.make_archive(OUT_DIR.rstrip("/"), "zip", OUT_DIR)
    say("")
    say(f"Finished: {len(done)} positions analysed in total ({state['new']} in this session), "
        f"top level {top:,} nodes per move, {len(ladder)} levels.")
    if state["failed"]:
        say(f"Positions that failed twice and were skipped: {state['failed']}")
    say(f"Results folder: {OUT_DIR}")
    say(f"Zipped copy:    {archive}")
    return info

In [ ]:
info = run()

## What is in the output files

**`lc0_evals.csv`** has one row per position, move and ladder level.

| column | meaning |
|---|---|
| `pos_id` | the position's place in the seeded order (0 = first) |
| `move` | the legal move that was played, in UCI notation |
| `level`, `target_nodes` | the ladder step and the node count asked for |
| `nodes` | the node count Lc0 actually reached (usually a little above the target) |
| `wl` | wins minus losses for the player who made the move, from -1 to 1. Expected score = (1 + `wl`) / 2 |
| `w`, `d`, `l` | win, draw and loss chances in thousandths, for the player who made the move |
| `mate` | filled in when Lc0 has proven a forced mate (positive = the mover wins) |
| `terminal` | `checkmate`, `stalemate` or `insufficient_material` when the move ends the game (no search needed) |
| `secs` | seconds spent on this move's ladder up to this level |

**`positions_analyzed.txt`** lists the finished positions (tab-separated: `pos_id`, line number in `optimal_positions.txt`, number of legal moves, FEN). The Stockfish script reads this file.

**`positions_order.csv`** is the seeded order itself (first 5,000 positions), with the reason for any position that is left out.

**`run_info.json`** records the settings, the ladder, the calibration table and the timings.

To continue a run in a new session, add the earlier output as input data and put its folder in `RESUME_DIR`.